# LCD test (LCD1602 with I2C backpack)

**Wiring** (the backpack is the small board soldered to the back of the LCD, with 4 pins):
| LCD backpack pin | Connect to |
|---|---|
| GND | GND |
| VCC | **5V** (Pi pin 2 or 4) |
| SDA | Pi pin 3 (GPIO 2), shared with the ADS7830 |
| SCL | Pi pin 5 (GPIO 3), shared with the ADS7830 |

I2C is a shared bus, so the LCD and the ADS7830 both hang off the same two pins. That's fine.

If the screen lights up but shows no text, or only solid blocks, turn the small blue contrast screw on the backpack with a screwdriver.

## 1. Setup

In [ ]:
import time
from smbus2 import SMBus
from RPLCD.i2c import CharLCD

LCD_ADDRESSES = (0x27, 0x3f, 0x20, 0x38)   # common addresses for PCF8574 backpacks


def find_lcd():
    """Return the I2C address the LCD backpack answers on, or None."""
    with SMBus(1) as bus:
        for addr in LCD_ADDRESSES:
            try:
                bus.read_byte(addr)
                return addr
            except OSError:
                pass
    return None


def show_speed(lcd, speed):
    """Write a speed value as an integer on line 2, padded so old digits are wiped."""
    lcd.cursor_pos = (1, 0)
    lcd.write_string(f"Speed: {int(speed):4d}    ")


# Re-running this cell is safe
try:
    lcd.close(clear=True)
except NameError:
    pass
addr = find_lcd()
if addr is None:
    raise RuntimeError("No LCD found on I2C. Check wiring (SDA pin 3, SCL pin 5, VCC 5V, GND).")
lcd = CharLCD(i2c_expander="PCF8574", address=addr, port=1, cols=16, rows=2)
print(f"LCD found at address 0x{addr:02x}")

## 2. Say hello

In [ ]:
lcd.clear()
lcd.write_string("Hello from Pi!")

## 3. Fake speed counter
Counts up on line 2 so you can see numbers updating. Press stop (■) to end.

In [ ]:
lcd.clear()
lcd.write_string("LCD test")
speed = 0
try:
    while True:
        show_speed(lcd, speed)
        speed = (speed + 37) % 1000
        time.sleep(0.5)
except KeyboardInterrupt:
    lcd.clear()
    lcd.write_string("Stopped")

In [ ]:
lcd.clear()

## Troubleshooting
- **"No LCD found"**: wrong wiring, or the backpack is on an unusual address. Run `!i2cdetect -y 1` in a cell; the LCD should show up as a second address next to `4b`.
- **Backlight on, no text**: adjust the contrast screw on the backpack.
- **Garbage characters**: the backpack is wired to the LCD in a non-standard order. Tell me and I'll change the pin map.